# Hub 2 - Modeling: misfit and Kalman calibration

Compares CTSM's simulated gross primary production (GPP) against what the NEON
tower measured, scores the misfit, then applies the project's Kalman
calibration and scores it again.

**The path:** read model output, fetch matching tower observations, join them
at monthly resolution, measure the misfit, calibrate, measure again.

**Prerequisites.** A completed simulation for the site below. If you have
none, `Getting_Started_CTSM_NEON.ipynb` produces one; `Walkthrough.ipynb`
covers the read-and-compare path in more detail. Observations are public and
need no credentials.

> **A trap worth knowing before you start.** The container ships a copy of
> `analytics_modules` at `/opt/analytics_modules`, which shadows a repository
> checkout on `sys.path`. If you are editing the library and your changes seem
> to have no effect, set `PYTHONPATH` to your checkout.

## 1. Model output

**Monthly, not half-hourly.** Observed GPP is not measured directly: towers
measure the *net* carbon flux, and photosynthesis is separated out by
estimating respiration and subtracting. When that estimate runs high the
arithmetic returns a negative value, which is physically impossible for a
gross flux. A quarter to a third of half-hourly observed GPP is negative at
every site. Averaging to monthly cancels it, because the errors are symmetric
noise around a small true value. That is why this Hub compares monthly series.
The reasoning and the alternatives that were tried and rejected are in
`docs/decisions/005-observed-gpp-comparison/`.

`open_ctsm_hist` works out which archive layout the wrapper used, whether the
stream names are current (`h0a`) or legacy (`h0`), and which NetCDF variant
the files are in. Asking for one variable rather than all 623 takes the read
from about two minutes to about ten seconds.

In [ ]:
import os

import pandas as pd

from analytics_modules import (
    calibrate_and_evaluate,
    evaluate_fit,
    find_ctsm_hist_files,
    monthly_observed_gpp,
    observed_gpp_coverage,
    open_ctsm_hist,
    residuals_plots,
    summarize_fit,
    time_series_comparison,
)

# KONZ is the default because it is the only one of the five sample sites with
# a complete observation record. See the coverage table in section 2.
SITE = "KONZ"

print("CTSM_OUTPUT_ROOT =", os.environ.get("CTSM_OUTPUT_ROOT", "/home/user (default)"))

try:
    month_files = find_ctsm_hist_files(SITE, stream="monthly")
    HAVE_OUTPUT = True
    print(f"Found {len(month_files)} monthly history files for {SITE}.")
except FileNotFoundError as exc:
    HAVE_OUTPUT = False
    print(f"No model output for {SITE} yet. The rest of this notebook needs it.\n")
    print("Run a simulation, or point CTSM_OUTPUT_ROOT at an existing archive.")
    print("Searched:\n", exc)

### Placing the model in time

**CTSM stamps monthly files at the start of the *next* month.**
`KONZ.transient.clm2.h0a.2018-07.nc` holds July's average but reports
`mcdate = 20180801`. An index built from `mcdate` is a month late, which is
easy to miss because the result still looks like a plausible seasonal cycle,
just displaced. Measured here it costs about 0.1 of correlation.

The reader attaches a `month` coordinate read from each filename, so the label
travels with the values. Build the index from that, never from `mcdate`.

In [ ]:
if HAVE_OUTPUT:
    monthly = open_ctsm_hist(SITE, stream="monthly", variables=["GPP"])
    model_gpp = pd.Series(
        monthly["GPP"].squeeze().values,
        index=pd.PeriodIndex(monthly["month"].values, freq="M"),
        name="model",
    ).sort_index()

    print(f"{len(model_gpp)} months, {model_gpp.index.min()} to {model_gpp.index.max()}")
    print("model GPP units:", monthly["GPP"].attrs.get("units"))

## 2. Tower observations

From the NCAR/NEON evaluation files: public, no credentials, one file per
site-month from 2018-01 to 2021-09.

Two things are handled for you, because each is a mistake that would otherwise
be made once per notebook.

- **Units.** Observations are in micromoles of CO2 per square metre per
  second; model GPP is in grams of carbon. The conversion is 12.011e-6.
  Getting it wrong is a five-order-of-magnitude error that reads as
  catastrophic model failure rather than as a bug, so conversion is the
  default.
- **Missing data.** 18% of site-months contain no GPP at all, and the quality
  flag does not tell you: some files report `GPP_fqc = 0` ("measured") across
  every timestep while every value is absent. Coverage is derived from the
  values, never the flag, and empty months are omitted rather than returned as
  missing, so nothing can quietly average over them.

In [ ]:
coverage = observed_gpp_coverage()
print(coverage[["months_with_gpp", "months_possible", "mean_negative_fraction"]].to_string())

**Only KONZ has all 45 months.** ABBY has 28. A comparison across all five
sites is limited to the months every site has, which is far fewer than it
appears. That is why this notebook defaults to KONZ, and it is a real
constraint on any multi-site work.

`low_signal` marks months whose mean is still below zero after averaging. They
are reported rather than clamped: all fall in the dormant season and sit
within 0.1 of zero in observation units, where model and tower are both
indistinguishable from zero. Hiding them would misrepresent the uncertainty.

In [ ]:
observed = monthly_observed_gpp(SITE)
observed.index = observed.index.to_period("M")

print(f"{len(observed)} months with data")
print(f"low_signal months: {int(observed.low_signal.sum())}")
print(observed[["gpp", "coverage", "negative_fraction", "low_signal"]].head(3).to_string())

## 3. Join

Both series are monthly and in the same units, so they join directly. The
column names follow the library's convention: observations under the variable
name, the model under `sim_<variable>`, and the calibrated model under
`cali_sim_<variable>`.

In [ ]:
if HAVE_OUTPUT:
    joined = pd.DataFrame({"GPP": observed["gpp"], "sim_GPP": model_gpp}).dropna()
    joined.index = joined.index.to_timestamp()
    joined = joined.reset_index().rename(columns={"index": "time"})

    print(f"overlapping months: {len(joined)}  "
          f"({joined.time.min():%Y-%m} to {joined.time.max():%Y-%m})")
    print(f"observed mean: {joined.GPP.mean():.3e} gC/m^2/s")
    print(f"model mean:    {joined.sim_GPP.mean():.3e} gC/m^2/s")
    print(f"model / observed: {joined.sim_GPP.mean() / joined.GPP.mean():.2f}")

## 4. Misfit before calibration

`residuals_plots` gives the diagnostic picture: observed against predicted,
the residuals over time, and their distribution. Its text conclusion flags
bias, error size relative to the observed spread, non-normal residuals, and
heteroscedasticity.

One thing to know when reading the numbers below against those in section 6:
**the two functions use opposite sign conventions for bias.**
`residuals_plots` reports the mean of `observed - predicted`, while
`evaluate_fit` reports the mean of `predicted - observed`. Same magnitude,
opposite sign.

In [ ]:
if HAVE_OUTPUT:
    fig, residuals, metrics, conclusion = residuals_plots(
        joined["GPP"], joined["sim_GPP"], bins=40, savepath=None,
    )
    print(conclusion)

## 5. Kalman calibration

The filter learns a time-varying bias and gain that map the model onto the
observations. State is `[bias, gain]` evolving as a random walk, with adaptive
observation noise and an optional smoother. It does **not** touch CTSM: it
takes the two series as arrays and learns a correction after the fact. A true
ensemble Kalman filter, which would run the model forward from a state and
cycle, was the intended design and was deferred for time rather than rejected
on technical grounds.

**What gets scored matters here.** The filter produces three series:

- the **one-step-ahead prediction**, the calibrated model at each month using
  only observations from earlier months. This is what `cali_sim_GPP` holds and
  what the post-calibration metrics are computed on.
- the **posterior fit**, which has already absorbed the observation it is then
  compared against. It is reported separately and labelled in-sample, because
  scoring it would overstate the improvement by construction.
- the **smoothed fit**, two-sided and therefore also in-sample.

If the calibration ever returns a near-perfect fit with a dead gain, that
means the bias term reproduced the observations on its own and the model was
ignored. That is reported as a failure rather than as a good result.

In [ ]:
if HAVE_OUTPUT:
    calibrated, summary = calibrate_and_evaluate(joined, "GPP")

    print("\nout-of-sample (reported):", {k: f"{v:.4e}" if isinstance(v, float) else v
                                           for k, v in summary["post_metrics"].items()
                                           if k in ("rmse", "mae", "r2", "bias")})
    print("in-sample posterior:     ", {k: f"{v:.4e}" if isinstance(v, float) else v
                                        for k, v in summary["posterior_metrics"].items()
                                        if k in ("rmse", "mae", "r2", "bias")})

## 6. What kind of misfit is it?

R-squared, RMSE, MAE and bias measure *how far* the model is from the
observations. They cannot say *why*. One worse RMSE could be an amplitude
error, a seasonal peak in the wrong month, noise, or a handful of bad months.

Scoring the seasonal cycle and the year-to-year variability separately turns
one undifferentiated number into a diagnosis:

- **seasonal phase shift** - months between the model's peak and the observed
  peak. Positive means the model peaks late.
- **seasonal amplitude ratio** - the model's seasonal swing over the observed
  one. Above 1 means the model's seasons are too strong.
- **interannual standard deviation ratio and anomaly correlation** - whether
  the model has the same good and bad years, and at the right strength.

These are ILAMB's definitions, so the scores are comparable to published ones,
but the ILAMB package itself is deliberately not used: its value is its
curated global reference data, and this project evaluates against tower
observations instead.

In [ ]:
if HAVE_OUTPUT:
    fit_raw = evaluate_fit(calibrated, "GPP", "sim_GPP")
    fit_cal = evaluate_fit(calibrated, "GPP", "cali_sim_GPP")

    rows = ["R2", "RMSE", "MAE", "Bias", "seasonal_phase_shift_months",
            "seasonal_amplitude_ratio", "iav_std_ratio", "iav_anomaly_r"]
    table = pd.DataFrame({"model": fit_raw, "model + Kalman": fit_cal}).loc[rows]
    print(table.to_string(float_format=lambda v: f"{v:.4g}"))

    print("\nmodel:         ", summarize_fit(fit_raw, "CLM"))
    print("\nmodel + Kalman:", summarize_fit(fit_cal, "the calibrated model"))

In [ ]:
if HAVE_OUTPUT:
    time_series_comparison(calibrated, f"{SITE} monthly GPP", "GPP")

## What this shows, and what it does not

**Shows.** The Hub runs end to end on native output with no credentials: a
simulation is read, matching tower observations are fetched, the two are
joined in consistent units at a resolution where the observation artifacts
cancel, the misfit is measured and attributed, and the Kalman calibration is
applied and scored honestly.

**On this data the calibration does not improve the out-of-sample fit, and
section 6 says why.** The raw model already peaks in the right month; its
problem is amplitude, a seasonal swing about 1.45x the observed. The
calibration does not fix that. It overshoots, damping the swing to about 0.43x
the observed, and introduces a one-month lag, because a filter learning a
time-varying correction from earlier months necessarily trails a seasonal
cycle. RMSE is unchanged and correlation falls.

That is a diagnosis rather than a verdict, and it is the kind of statement the
plain magnitude metrics could not have produced: R-squared alone would only
have said "worse". The in-sample posterior does score better, which is exactly
why it is reported separately - read it as how well the filter can fit what it
has already seen, not as model skill.

**Does not.**

- **One site, and not a live run.** KONZ is the only sample site with a
  complete observation record; the other four have gaps, worst at ABBY with 28
  of 45 months. Numbers produced from the staged reference copies also come
  from an older model generation, so rerun this against your own output before
  quoting them.
- **No agreed tolerance.** Nothing here passes or fails; how close is close
  enough is not a settled question for this project.
- **Under four years.** The interannual scores come from 45 months, a thin
  basis for year-to-year statistics. The metrics flag this themselves via
  `short_record`.
- **Gross against partitioned.** Model GPP is gross by construction, while
  tower GPP is separated out of a measured net flux. Even a perfect model
  would not match exactly.